<!-- SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: OpenMDW-1.1 -->

# Cosmos3 Generator Audiovisual with TensorRT-LLM

This notebook calls already-running TensorRT-LLM VisualGen servers with direct `curl` requests from Python.

The examples are split into Cosmos3-Nano, Cosmos3-Super, and Cosmos3-Edge sections. Each section is self-contained, so you can run just one. The notebook covers TensorRT-LLM's stable server flow for text-to-image, text-to-video and image-to-video with or without synchronized audio, and video-to-video generation.

`Cosmos3-Edge` is the compact single-GPU checkpoint. TensorRT-LLM serves it for text-to-image, text-to-video, and image-to-video only: Edge has no audio tower, its action weights are not served by this pipeline, and video-to-video is validated for Nano and Super only.


## 1. Prerequisites

Use a running TensorRT-LLM server with Cosmos3 VisualGen audio and video-to-video support, and set endpoint environment variables before the setup cell if you are not using the local default. Text-to-image uses `/v1/images/generations`; video generation uses `/v1/videos/sync`. Image-to-video uploads multipart `image_reference`, while video-to-video uploads multipart `video_reference`.

Install the `ffmpeg` CLI in the server environment. This notebook explicitly requests browser-compatible MP4, so TensorRT-LLM returns an actionable error if the encoder is unavailable; `ffmpeg` also muxes synchronized audio.

Generator requires the Guardrail. Request access to the gated [nvidia/Cosmos-1.0-Guardrail](https://huggingface.co/nvidia/Cosmos-1.0-Guardrail) HF repository before running these examples. TensorRT-LLM loads guardrails by default; to disable them, set `TRTLLM_DISABLE_COSMOS3_GUARDRAILS=1` before starting the server or set `use_guardrails` to `False` in the request `extra_params`.

```bash
export COSMOS3_TRTLLM_BASE_URL=http://localhost:8000
export COSMOS3_TRTLLM_NANO_BASE_URL=http://localhost:8000
export COSMOS3_TRTLLM_SUPER_BASE_URL=http://localhost:8000
export COSMOS3_TRTLLM_EDGE_BASE_URL=http://localhost:8000
export COSMOS3_TRTLLM_API_KEY=tensorrt_llm
```


## 2. Start the Server

Run the TensorRT-LLM VisualGen server before running the request cells. The config YAMLs below come from TensorRT-LLM's Cosmos3 support. To build a checkout from source, follow NVIDIA's [Build from Source](https://nvidia.github.io/TensorRT-LLM/installation/build-from-source.html) guide and then run the commands below from that checkout. Install `ffmpeg` in that environment before serving (`apt-get update && apt-get install -y ffmpeg` on Ubuntu/Debian images).

### Cosmos3-Nano

From inside the TensorRT-LLM checkout — the directory that contains `examples/` — with TensorRT-LLM installed in the active environment:

```bash
export TRTLLM_ROOT="${TRTLLM_ROOT:-$PWD}"

trtllm-serve nvidia/Cosmos3-Nano \
  --visual_gen_args "$TRTLLM_ROOT/examples/visual_gen/configs/cosmos3-nano-1gpu.yaml" \
  --port 8000
```

### Cosmos3-Super

`Cosmos3-Super` uses the four-GPU config from TensorRT-LLM. The config sets `cfg_size=2`, `ulysses_size=2`, and `parallel_vae_size=4`, so launch exactly four processes.

```bash
export TRTLLM_ROOT="${TRTLLM_ROOT:-$PWD}"

torchrun --nproc_per_node=4 -m tensorrt_llm.commands.serve \
  nvidia/Cosmos3-Super \
  --visual_gen_args "$TRTLLM_ROOT/examples/visual_gen/configs/cosmos3-super-4gpu.yaml" \
  --port 8000
```

### Cosmos3-Edge

`Cosmos3-Edge` is the compact 4B Nemotron-dense checkpoint and fits on a single GPU. Its 480p-native generation defaults — 832x480 with 121 frames, 50 UniPC steps on the checkpoint-declared native flow schedule, guidance 5.0, flow shift 3.0 — are read from the checkpoint, so Edge needs no config override:

```bash
trtllm-serve nvidia/Cosmos3-Edge --port 8000
```

TensorRT-LLM serves Edge for text-to-image, text-to-video, and image-to-video. Text-to-image goes to `/v1/images/generations` with `output_type="image"`; the two video modes go to `/v1/videos/sync`. Edge has no audio tower, so `enable_audio` is unavailable, and the checkpoint's action weights are not served by this pipeline. Requests outside the model card's validated envelope (256p/480p, 50-150 frames, 12-30 FPS) still run and log an advisory line.

TensorRT-LLM exposes `/health` when the server is ready. This notebook sends Cosmos3 model-specific controls through `extra_params`, so use a TensorRT-LLM release that includes the Cosmos3 VisualGen API schema.


## 3. Configure Paths and Endpoints

This setup cell only configures repo/output paths and TensorRT-LLM endpoint settings.


In [ ]:
from pathlib import Path
import os


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "README.md").exists() and (path / "cookbooks").exists():
            return path
    return start


COSMOS_ROOT = find_repo_root(Path.cwd().resolve())
COSMOS3_AUDIOVISUAL_ROOT = COSMOS_ROOT / "cookbooks" / "cosmos3" / "generator" / "audiovisual"
COSMOS3_AUDIOVISUAL_OUTPUT_ROOT = Path(
    os.environ.get("COSMOS3_AUDIOVISUAL_OUTPUT_ROOT", COSMOS3_AUDIOVISUAL_ROOT / "outputs" / "notebooks")
).resolve()
DEFAULT_TRTLLM_BASE_URL = os.environ.get("COSMOS3_TRTLLM_BASE_URL", "http://localhost:8000")
TRTLLM_ENDPOINTS = {
    "Cosmos3-Nano": os.environ.get("COSMOS3_TRTLLM_NANO_BASE_URL", DEFAULT_TRTLLM_BASE_URL),
    "Cosmos3-Super": os.environ.get("COSMOS3_TRTLLM_SUPER_BASE_URL", DEFAULT_TRTLLM_BASE_URL),
    "Cosmos3-Edge": os.environ.get("COSMOS3_TRTLLM_EDGE_BASE_URL", DEFAULT_TRTLLM_BASE_URL),
}

os.environ["COSMOS3_AUDIOVISUAL_OUTPUT_ROOT"] = str(COSMOS3_AUDIOVISUAL_OUTPUT_ROOT)
os.environ.setdefault("COSMOS3_TRTLLM_API_KEY", "tensorrt_llm")

print("COSMOS_ROOT:", COSMOS_ROOT)
print("COSMOS3_AUDIOVISUAL_OUTPUT_ROOT:", COSMOS3_AUDIOVISUAL_OUTPUT_ROOT)
for model, endpoint in TRTLLM_ENDPOINTS.items():
    print(f"{model} endpoint: {endpoint}")


## 4. Verify Endpoint Configuration


In [ ]:
from urllib.parse import urlparse


def api_root_url(base_url: str) -> str:
    normalized = base_url.rstrip("/")
    if not normalized.endswith("/v1"):
        normalized = f"{normalized}/v1"
    return normalized


def server_root_url(base_url: str) -> str:
    normalized = base_url.rstrip("/")
    if normalized.endswith("/v1"):
        normalized = normalized[:-3]
    return normalized.rstrip("/")


def health_url(base_url: str) -> str:
    return f"{server_root_url(base_url)}/health"


def video_api_url(base_url: str) -> str:
    return f"{api_root_url(base_url)}/videos/sync"


def image_api_url(base_url: str) -> str:
    return f"{api_root_url(base_url)}/images/generations"


for model, base_url in TRTLLM_ENDPOINTS.items():
    parsed = urlparse(api_root_url(base_url))
    print(model)
    print("  api root:", api_root_url(base_url))
    print("  health:", health_url(base_url))
    print("  images:", image_api_url(base_url))
    print("  videos sync:", video_api_url(base_url))
    print("  scheme:", parsed.scheme)
    print("  host:", parsed.netloc)


## 5. Preview Available Inputs


In [ ]:
from pathlib import Path
import json
from IPython.display import Image, display

assets_dir = COSMOS3_AUDIOVISUAL_ROOT / "assets"
for prompt_dir in sorted((assets_dir / "prompts").iterdir()):
    if not prompt_dir.is_dir():
        continue
    print(f"{prompt_dir.relative_to(assets_dir)}:")
    for prompt_path in sorted(prompt_dir.glob("*.json")):
        data = json.loads(prompt_path.read_text())
        caption = (
            data.get("temporal_caption")
            or data.get("comprehensive_t2i_caption")
            or data.get("extra", {}).get("prompt", "")
        )
        print(f"  {prompt_path.name}: {caption[:180]}{'...' if len(caption) > 180 else ''}")
    print()

for image_dir in sorted((assets_dir / "images").iterdir()):
    if not image_dir.is_dir():
        continue
    print(f"{image_dir.relative_to(assets_dir)}:")
    for image_path in sorted(image_dir.iterdir()):
        if image_path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".bmp"}:
            print(f"  {image_path.name}")
            display(Image(filename=str(image_path), width=420))

videos_dir = assets_dir / "videos"
print(f"{videos_dir.relative_to(assets_dir)}:")
for video_path in sorted(videos_dir.iterdir()):
    if video_path.suffix.lower() in {".mp4", ".avi"}:
        print(f"  {video_path.name} ({video_path.stat().st_size // 1024} KB)")


## 6. Define Asset Sets, Payload Helpers, Request Helpers, and Viewer Helpers


In [ ]:
import base64
import html
import json
import os
import subprocess
import time
import urllib.error
import urllib.request
from pathlib import Path
from IPython.display import HTML, Image, display


def api_root_url(base_url: str) -> str:
    normalized = base_url.rstrip("/")
    if not normalized.endswith("/v1"):
        normalized = f"{normalized}/v1"
    return normalized


def server_root_url(base_url: str) -> str:
    normalized = base_url.rstrip("/")
    if normalized.endswith("/v1"):
        normalized = normalized[:-3]
    return normalized.rstrip("/")


def health_url(base_url: str) -> str:
    return f"{server_root_url(base_url)}/health"


def video_api_url(base_url: str) -> str:
    return f"{api_root_url(base_url)}/videos/sync"


def image_api_url(base_url: str) -> str:
    return f"{api_root_url(base_url)}/images/generations"


IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

FIXED_SAMPLING = {
    "num_steps": 35,
    "guidance": 6.0,
    "fps": 24,
    "num_frames": 189,
    "max_sequence_length": 4096,
    "resolution": "720",
    "aspect_ratio": "16,9",
    "seed": 0,
}

# Cosmos3-Edge is 480p-native and has no audio tower, so it runs a different
# shape from Nano/Super. These are the checkpoint's deployed TensorRT-LLM
# defaults, sent explicitly so each request is self-describing. Flow shift (3.0)
# rides the checkpoint-declared native flow schedule and is not a request field.
EDGE_VIDEO_SAMPLING = {
    "num_steps": 50,
    "guidance": 5.0,
    "fps": 24,
    "num_frames": 121,
    "max_sequence_length": 4096,
    "resolution": "480",
    "aspect_ratio": "16,9",
    "seed": 0,
}

# Edge text-to-image runs at its native 640x640 with full-range CFG, through the
# image generation endpoint in native image mode. That path carries no frame or
# frame-rate fields, so this table has none either.
EDGE_T2I_SAMPLING = {
    "num_steps": 50,
    "guidance": 4.0,
    "max_sequence_length": 4096,
    "resolution": "640",
    "aspect_ratio": "1,1",
    "seed": 0,
}

TRTLLM_EXTRA_PARAMS = {
    "use_resolution_template": False,
    "use_duration_template": False,
    "use_system_prompt": False,
    "use_guardrails": True,
}
# TensorRT-LLM uses the image endpoint for T2I and the synchronous video
# endpoint for T2V, I2V, and V2V.
ASSET_SETS = {
    "t2i_nano": {
        "model": "Cosmos3-Nano",
        "mode": "text2image",
        "prompt": "assets/prompts/text2image/robot_draping.json",
    },
    "t2v_nano": {
        "model": "Cosmos3-Nano",
        "mode": "text2video",
        "prompt": "assets/prompts/text2video/robot_kitchen.json",
    },
    "t2av_nano": {
        "model": "Cosmos3-Nano",
        "mode": "text2video",
        "prompt": "assets/prompts/text2video/robot_pouring_water_audio.json",
        "enable_audio": True,
    },
    "i2v_nano": {
        "model": "Cosmos3-Nano",
        "mode": "image2video",
        "prompt": "assets/prompts/image2video/humanoid_robot.json",
        "image": "assets/images/image2video/humanoid_robot.jpg",
    },
    "i2av_nano": {
        "model": "Cosmos3-Nano",
        "mode": "image2video",
        "prompt": "assets/prompts/image2video/coastal_road_audio.json",
        "image": "assets/images/image2video/coastal_road_audio.jpg",
        "enable_audio": True,
    },
    "v2v_nano": {
        "model": "Cosmos3-Nano",
        "mode": "video2video",
        "prompt": "assets/prompts/image2video/car_driving.json",
        "negative_prompt": "assets/negative_prompts/image2video/neg_prompt.json",
        "video": "assets/videos/car_driving_plain.mp4",
        "extra_params": {
            "use_system_prompt": True,
            "condition_video_latent_indexes": [0, 1],
            "condition_video_keep": "first",
        },
    },
    "t2i_super": {
        "model": "Cosmos3-Super",
        "mode": "text2image",
        "prompt": "assets/prompts/text2image/robot_draping.json",
    },
    "t2v_super": {
        "model": "Cosmos3-Super",
        "mode": "text2video",
        "prompt": "assets/prompts/text2video/robot_kitchen.json",
    },
    "t2av_super": {
        "model": "Cosmos3-Super",
        "mode": "text2video",
        "prompt": "assets/prompts/text2video/robot_pouring_water_audio.json",
        "enable_audio": True,
    },
    "i2v_super": {
        "model": "Cosmos3-Super",
        "mode": "image2video",
        "prompt": "assets/prompts/image2video/humanoid_robot.json",
        "image": "assets/images/image2video/humanoid_robot.jpg",
    },
    "i2av_super": {
        "model": "Cosmos3-Super",
        "mode": "image2video",
        "prompt": "assets/prompts/image2video/coastal_road_audio.json",
        "image": "assets/images/image2video/coastal_road_audio.jpg",
        "enable_audio": True,
    },
    "v2v_super": {
        "model": "Cosmos3-Super",
        "mode": "video2video",
        "prompt": "assets/prompts/image2video/car_driving.json",
        "negative_prompt": "assets/negative_prompts/image2video/neg_prompt.json",
        "video": "assets/videos/car_driving_plain.mp4",
        "extra_params": {
            "use_system_prompt": True,
            "condition_video_latent_indexes": [0, 1],
            "condition_video_keep": "first",
        },
    },
    "t2i_edge": {
        "model": "Cosmos3-Edge",
        "mode": "text2image",
        "prompt": "assets/prompts/text2image/robot_draping.json",
        "sampling": EDGE_T2I_SAMPLING,
        # Native text-to-image. Without this the request runs the video path,
        # which defaults the negative prompt to Cosmos3's video negative prompt
        # (motion and frame-to-frame artifact terms) instead of leaving it empty.
        "extra_params": {"output_type": "image"},
    },
    "t2v_edge": {
        "model": "Cosmos3-Edge",
        "mode": "text2video",
        "prompt": "assets/prompts/text2video/robot_kitchen.json",
        "sampling": EDGE_VIDEO_SAMPLING,
    },
    "i2v_edge": {
        "model": "Cosmos3-Edge",
        "mode": "image2video",
        "prompt": "assets/prompts/image2video/car_driving_edge.json",
        "image": "assets/images/image2video/car_driving.jpg",
        "sampling": EDGE_VIDEO_SAMPLING,
    },
}


def asset_path(relative_path: str) -> Path:
    path = COSMOS3_AUDIOVISUAL_ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(path)
    return path.resolve()


def compact_json_file(path: Path) -> str:
    return json.dumps(json.loads(path.read_text()), ensure_ascii=True, separators=(",", ":"))


def payload_dimensions(payload: dict) -> tuple[int, int]:
    if payload.get("resolution") == "720" and payload.get("aspect_ratio") == "16,9":
        return 720, 1280
    if payload.get("resolution") == "480" and payload.get("aspect_ratio") == "16,9":
        return 480, 832
    if payload.get("resolution") == "640" and payload.get("aspect_ratio") == "1,1":
        return 640, 640
    if payload.get("resolution") == "256" and payload.get("aspect_ratio") == "16,9":
        return 192, 320
    raise ValueError(f"Unsupported payload resolution/aspect ratio: {payload.get('resolution')} {payload.get('aspect_ratio')}")


def resolve_payload_path(payload_path: Path, value: str) -> Path:
    path = Path(value)
    if path.is_absolute():
        return path
    return (payload_path.parent / path).resolve()


def create_payload(use_case: str, *, backend: str = "trt_llm") -> tuple[Path, Path, str]:
    spec = ASSET_SETS[use_case]
    payload_dir = Path(os.environ["COSMOS3_AUDIOVISUAL_OUTPUT_ROOT"]) / backend / "payloads" / use_case
    output_dir = Path(os.environ["COSMOS3_AUDIOVISUAL_OUTPUT_ROOT"]) / backend / use_case
    payload_dir.mkdir(parents=True, exist_ok=True)
    output_dir.mkdir(parents=True, exist_ok=True)

    prompt_path = asset_path(spec["prompt"])
    payload_path = payload_dir / f"{use_case}.json"
    extra_params = dict(TRTLLM_EXTRA_PARAMS)
    extra_params["enable_audio"] = bool(spec.get("enable_audio", False))
    extra_params.update(spec.get("extra_params", {}))
    if spec["mode"] == "text2image":
        extra_params["output_type"] = "image"
    payload = {
        "model_mode": spec["mode"],
        "name": use_case,
        "prompt": compact_json_file(prompt_path),
        "extra_params": extra_params,
        **spec.get("sampling", FIXED_SAMPLING),
    }
    if spec["mode"] == "text2image":
        payload.pop("num_frames", None)
        payload.pop("fps", None)
    else:
        negative_prompt_path = asset_path(
            spec.get("negative_prompt") or f"assets/negative_prompts/{spec['mode']}/neg_prompt.json"
        )
        payload["negative_prompt"] = compact_json_file(negative_prompt_path)
    if spec["mode"] == "image2video":
        image_path = asset_path(spec["image"])
        payload["vision_path"] = os.path.relpath(image_path, payload_path.parent)
    if spec["mode"] == "video2video":
        video_path = asset_path(spec["video"])
        payload["video_path"] = os.path.relpath(video_path, payload_path.parent)

    payload_path.write_text(json.dumps(payload, indent=2) + "\n")

    os.environ[f"COSMOS3_{backend.upper()}_{use_case.upper()}_INPUT"] = str(payload_path)
    os.environ[f"COSMOS3_{backend.upper()}_{use_case.upper()}_OUTPUT"] = str(output_dir)

    print(f"model:   {spec['model']}")
    print(f"payload: {payload_path}")
    print(f"output:  {output_dir}")
    print(f"prompt:  {prompt_path.relative_to(COSMOS_ROOT)}")
    if spec["mode"] == "text2image":
        print("note:   TensorRT-LLM Cosmos3 text-to-image uses /v1/images/generations")
    if "vision_path" in payload:
        image_display_path = resolve_payload_path(payload_path, payload["vision_path"])
        print(f"image:   {image_display_path.relative_to(COSMOS_ROOT)}")
        display(Image(filename=str(image_display_path), width=420))
    if "video_path" in payload:
        video_display_path = resolve_payload_path(payload_path, payload["video_path"])
        print(f"video:   {video_display_path.relative_to(COSMOS_ROOT)}")
        print("note:    condition_video_latent_indexes=[0, 1] consumes five input pixel frames")
    if payload["extra_params"]["enable_audio"]:
        print("audio:   enabled; ffmpeg is required on the server to mux it into MP4")
    preview_keys = ["model_mode", "name", "num_steps", "guidance"]
    if spec["mode"] != "text2image":
        preview_keys.extend(["fps", "num_frames"])
    preview_keys.extend(["max_sequence_length", "resolution", "aspect_ratio", "seed", "extra_params"])
    if "seconds" in payload:
        preview_keys.insert(5, "seconds")
    print(json.dumps({k: payload[k] for k in preview_keys}, indent=2))
    return payload_path, output_dir, spec["model"]


def check_trtllm_server(model: str, timeout_s: int = 1800, interval_s: int = 10) -> None:
    url = health_url(TRTLLM_ENDPOINTS[model])
    deadline = time.time() + timeout_s
    print(f"waiting for {model} server: {url}")
    last_error = None
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as response:
                if response.status == 200:
                    print(f"{model} server is ready")
                    return
                last_error = f"HTTP {response.status}"
        except (urllib.error.URLError, TimeoutError, OSError) as exc:
            last_error = str(exc)
        print(f"not ready yet: {last_error}")
        time.sleep(interval_s)
    raise TimeoutError(f"Timed out waiting for {model} server at {url}. Last error: {last_error}")


def build_trtllm_video_body(payload: dict) -> dict:
    height, width = payload_dimensions(payload)
    body = {
        "prompt": payload["prompt"],
        "size": f"{width}x{height}",
        "seconds": payload.get("seconds", payload["num_frames"] / payload["fps"]),
        "fps": payload["fps"],
        "num_frames": payload["num_frames"],
        "max_sequence_length": payload["max_sequence_length"],
        "seed": payload["seed"],
        "format": "mp4",
        "response_format": "file",
    }
    if payload.get("num_steps") is not None:
        body["num_inference_steps"] = payload["num_steps"]
    if payload.get("guidance") is not None:
        body["guidance_scale"] = payload["guidance"]
    if payload.get("negative_prompt") is not None:
        body["negative_prompt"] = payload["negative_prompt"]
    body["extra_params"] = payload["extra_params"]
    return body


def build_trtllm_image_body(payload: dict) -> dict:
    height, width = payload_dimensions(payload)
    body = {
        "prompt": payload["prompt"],
        "size": f"{width}x{height}",
        "format": "png",
        "response_format": "b64_json",
        "max_sequence_length": payload["max_sequence_length"],
        "seed": payload["seed"],
        "extra_params": payload["extra_params"],
    }
    if payload.get("num_steps") is not None:
        body["num_inference_steps"] = payload["num_steps"]
    if payload.get("guidance") is not None:
        body["guidance_scale"] = payload["guidance"]
    if payload.get("negative_prompt") is not None:
        body["negative_prompt"] = payload["negative_prompt"]
    return body


def _auth_headers() -> list[str]:
    api_key = os.environ.get("COSMOS3_TRTLLM_API_KEY", "")
    return ["-H", f"Authorization: Bearer {api_key}"] if api_key else []


def _require_mp4_response(header_text: str, content_path: Path) -> None:
    if "video/mp4" not in header_text.lower():
        raise RuntimeError(f"Expected video/mp4 response, got headers:\n{header_text}")
    head = content_path.read_bytes()[:16]
    if len(head) < 12 or head[4:8] != b"ftyp":
        raise RuntimeError("TensorRT-LLM returned a non-MP4 payload")


def post_video(*, payload_path: Path, payload: dict, output_stem: Path, model: str) -> Path:
    url = video_api_url(TRTLLM_ENDPOINTS[model])
    tmp_path = Path(f"{output_stem}.tmp")
    header_path = Path(f"{output_stem}.headers.txt")
    error_path = Path(f"{output_stem}.error.txt")
    for path in [tmp_path, header_path, error_path]:
        if path.exists():
            path.unlink()

    body = build_trtllm_video_body(payload)
    cmd = [
        "curl",
        "-sS",
        "--fail-with-body",
        "-X",
        "POST",
        url,
        "-D",
        str(header_path),
        "-H",
        "Accept: video/mp4",
    ]
    cmd += _auth_headers()

    if payload["model_mode"] in {"image2video", "video2video"}:
        form_body = dict(body)
        if "extra_params" in form_body:
            form_body["extra_params"] = json.dumps(form_body["extra_params"], separators=(",", ":"))
        for key, value in form_body.items():
            cmd += ["--form-string", f"{key}={value}"]
        if payload["model_mode"] == "image2video":
            reference_path = resolve_payload_path(payload_path, payload["vision_path"])
            reference_type = "image/jpeg"
            reference_field = "image_reference"
        else:
            reference_path = resolve_payload_path(payload_path, payload["video_path"])
            reference_type = "video/mp4"
            reference_field = "video_reference"
        cmd += ["-F", f"{reference_field}=@{reference_path};type={reference_type}"]
    else:
        cmd += ["-H", "Content-Type: application/json"]
        cmd += ["-d", json.dumps(body, separators=(",", ":"))]

    cmd += ["-o", str(tmp_path)]
    result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if result.returncode != 0:
        error_path.write_text((result.stdout or "") + (result.stderr or ""))
        raise RuntimeError(f"TensorRT-LLM request failed with exit code {result.returncode}; see {error_path}")

    header_text = header_path.read_text() if header_path.exists() else ""
    _require_mp4_response(header_text, tmp_path)
    output_path = output_stem.with_suffix(".mp4")
    if output_path.exists():
        output_path.unlink()
    tmp_path.replace(output_path)
    return output_path


def post_image(*, payload: dict, output_stem: Path, model: str) -> Path:
    url = image_api_url(TRTLLM_ENDPOINTS[model])
    response_path = Path(f"{output_stem}.response.json")
    error_path = Path(f"{output_stem}.error.txt")
    for path in [response_path, error_path]:
        if path.exists():
            path.unlink()

    cmd = [
        "curl",
        "-sS",
        "--fail-with-body",
        "-X",
        "POST",
        url,
        "-H",
        "Content-Type: application/json",
        "-d",
        json.dumps(build_trtllm_image_body(payload), separators=(",", ":")),
        "-o",
        str(response_path),
    ]
    cmd += _auth_headers()
    result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if result.returncode != 0:
        server_body = response_path.read_text() if response_path.exists() else ""
        error_path.write_text(server_body + (result.stdout or "") + (result.stderr or ""))
        raise RuntimeError(f"TensorRT-LLM request failed with exit code {result.returncode}; see {error_path}")

    response = json.loads(response_path.read_text())
    try:
        image_bytes = base64.b64decode(response["data"][0]["b64_json"], validate=True)
    except (KeyError, IndexError, TypeError, ValueError) as exc:
        raise RuntimeError(f"Unexpected TensorRT-LLM image response: {response}") from exc
    if not image_bytes.startswith(b"\x89PNG\r\n\x1a\n"):
        raise RuntimeError("TensorRT-LLM returned a non-PNG image payload")
    output_path = output_stem.with_suffix(".png")
    output_path.write_bytes(image_bytes)
    return output_path


def run_trtllm_payload(payload_path: Path, output_dir: str | Path, *, model: str) -> Path:
    payload_path = Path(payload_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    payload = json.loads(payload_path.read_text())
    output_stem = output_dir / payload["name"]
    endpoint = (
        image_api_url(TRTLLM_ENDPOINTS[model])
        if payload["model_mode"] == "text2image"
        else video_api_url(TRTLLM_ENDPOINTS[model])
    )
    print("endpoint:", endpoint)
    print("payload:", payload_path)
    print("output stem:", output_stem)
    if payload["model_mode"] == "image2video":
        print("input image:", resolve_payload_path(payload_path, payload["vision_path"]))
    if payload["model_mode"] == "video2video":
        print("input video:", resolve_payload_path(payload_path, payload["video_path"]))
    t0 = time.time()
    if payload["model_mode"] == "text2image":
        output_path = post_image(payload=payload, output_stem=output_stem, model=model)
    else:
        output_path = post_video(payload_path=payload_path, payload=payload, output_stem=output_stem, model=model)
    print(f"wrote {output_path} in {time.time() - t0:.1f}s")
    return output_path


def display_video(path: Path, *, width: int = 720) -> None:
    media_type = "video/mp4"
    data = base64.b64encode(path.read_bytes()).decode("ascii")
    label = html.escape(str(path))
    markup = f"""
<video controls playsinline preload="metadata" width="{width}" style="max-width: 100%; background: #000;">
  <source src="data:{media_type};base64,{data}" type="{media_type}">
</video>
<div style="font-family: monospace; font-size: 12px; margin-top: 4px;">{label}</div>
"""
    display(HTML(markup))


def view_run(output_dir: str | Path) -> None:
    output_dir = Path(output_dir)
    images = [path for path in sorted(output_dir.rglob("*.png"))]
    videos = [
        path
        for path in sorted(output_dir.rglob("*"))
        if path.suffix.lower() == ".mp4"
        and not path.name.endswith(("_preview.mp4", "_browser.mp4"))
    ]
    if not images and not videos:
        print(f"No generated media found under {output_dir}")
        return
    for src in images:
        print(f"source: {src} ({src.stat().st_size // 1024} KB)")
        display(Image(filename=str(src), width=720))
    for src in videos:
        print(f"source: {src} ({src.stat().st_size // 1024} KB)")
        display_video(src)


Run each use case top-to-bottom: create the JSON payload, wait for the matching server, run inference, then view the generated media. The Cosmos3-Nano and Cosmos3-Super sections are independent, so you can run just one.


# Cosmos3-Nano Examples

Use cases for the `Cosmos3-Nano` model. This section is self-contained; you can run it without the Cosmos3-Super section below.


## Nano: Text to Image

Nano text-to-image generation using a structured JSON prompt. TensorRT-LLM returns a base64-encoded PNG from `/v1/images/generations`.


In [ ]:
t2i_nano_payload, t2i_nano_output, t2i_nano_model = create_payload("t2i_nano")


### Run


In [ ]:
check_trtllm_server(t2i_nano_model)
run_trtllm_payload(t2i_nano_payload, t2i_nano_output, model=t2i_nano_model)


### View Results


In [ ]:
view_run(t2i_nano_output)


## Nano: Text to Video Without Audio

Nano text-to-video generation using a structured JSON prompt, with audio disabled.


In [ ]:
t2v_nano_payload, t2v_nano_output, t2v_nano_model = create_payload("t2v_nano")


### Run


In [ ]:
check_trtllm_server(t2v_nano_model)
run_trtllm_payload(t2v_nano_payload, t2v_nano_output, model=t2v_nano_model)


### View Results


In [ ]:
view_run(t2v_nano_output)


## Nano: Text to Video with Audio

Nano text-to-video generation with synchronized audio. The payload enables `enable_audio` in TensorRT-LLM's Cosmos3 `extra_params`; `ffmpeg` muxes the generated audio into the MP4 response.


In [ ]:
t2av_nano_payload, t2av_nano_output, t2av_nano_model = create_payload("t2av_nano")


### Run


In [ ]:
check_trtllm_server(t2av_nano_model)
run_trtllm_payload(t2av_nano_payload, t2av_nano_output, model=t2av_nano_model)


### View Results


In [ ]:
view_run(t2av_nano_output)


## Nano: Image to Video Without Audio

Nano image-to-video generation using its paired image asset, with audio disabled.


In [ ]:
i2v_nano_payload, i2v_nano_output, i2v_nano_model = create_payload("i2v_nano")


### Run


In [ ]:
check_trtllm_server(i2v_nano_model)
run_trtllm_payload(i2v_nano_payload, i2v_nano_output, model=i2v_nano_model)


### View Results


In [ ]:
view_run(i2v_nano_output)


## Nano: Image to Video with Audio

Nano image-to-video generation using its paired coastal-road image and synchronized audio prompt. The reference image is uploaded as multipart `image_reference`, and `enable_audio` is set in `extra_params`.


In [ ]:
i2av_nano_payload, i2av_nano_output, i2av_nano_model = create_payload("i2av_nano")


### Run


In [ ]:
check_trtllm_server(i2av_nano_model)
run_trtllm_payload(i2av_nano_payload, i2av_nano_output, model=i2av_nano_model)


### View Results


In [ ]:
view_run(i2av_nano_output)


## Nano: Video to Video

Nano video-to-video generation using a checked-in MP4 `video_reference`. TensorRT-LLM forwards the encoded video to each worker for NVDEC decoding. `condition_video_latent_indexes=[0, 1]` pins the first two output latent frames, which consumes the first five pixel frames of the reference.


In [ ]:
v2v_nano_payload, v2v_nano_output, v2v_nano_model = create_payload("v2v_nano")


### Run


In [ ]:
check_trtllm_server(v2v_nano_model)
run_trtllm_payload(v2v_nano_payload, v2v_nano_output, model=v2v_nano_model)


### View Results


In [ ]:
view_run(v2v_nano_output)


# Cosmos3-Super Examples

The same use cases for the larger `Cosmos3-Super` model. This section is self-contained; you can run it without the Cosmos3-Nano section above.


## Super: Text to Image

Super text-to-image generation using the same structured JSON prompt. TensorRT-LLM returns a base64-encoded PNG from `/v1/images/generations`.


In [ ]:
t2i_super_payload, t2i_super_output, t2i_super_model = create_payload("t2i_super")


### Run


In [ ]:
check_trtllm_server(t2i_super_model)
run_trtllm_payload(t2i_super_payload, t2i_super_output, model=t2i_super_model)


### View Results


In [ ]:
view_run(t2i_super_output)


## Super: Text to Video Without Audio

Super text-to-video generation using the same structured JSON prompt, with audio disabled.


In [ ]:
t2v_super_payload, t2v_super_output, t2v_super_model = create_payload("t2v_super")


### Run


In [ ]:
check_trtllm_server(t2v_super_model)
run_trtllm_payload(t2v_super_payload, t2v_super_output, model=t2v_super_model)


### View Results


In [ ]:
view_run(t2v_super_output)


## Super: Text to Video with Audio

Super text-to-video generation with synchronized audio using the same `enable_audio` request contract as Nano.


In [ ]:
t2av_super_payload, t2av_super_output, t2av_super_model = create_payload("t2av_super")


### Run


In [ ]:
check_trtllm_server(t2av_super_model)
run_trtllm_payload(t2av_super_payload, t2av_super_output, model=t2av_super_model)


### View Results


In [ ]:
view_run(t2av_super_output)


## Super: Image to Video Without Audio

Super image-to-video generation using its paired image asset, with audio disabled.


In [ ]:
i2v_super_payload, i2v_super_output, i2v_super_model = create_payload("i2v_super")


### Run


In [ ]:
check_trtllm_server(i2v_super_model)
run_trtllm_payload(i2v_super_payload, i2v_super_output, model=i2v_super_model)


### View Results


In [ ]:
view_run(i2v_super_output)


## Super: Image to Video with Audio

Super image-to-video generation using the paired coastal-road image and synchronized audio prompt.


In [ ]:
i2av_super_payload, i2av_super_output, i2av_super_model = create_payload("i2av_super")


### Run


In [ ]:
check_trtllm_server(i2av_super_model)
run_trtllm_payload(i2av_super_payload, i2av_super_output, model=i2av_super_model)


### View Results


In [ ]:
view_run(i2av_super_output)


## Super: Video to Video

Super video-to-video generation using the same reference-video request shape and conditioning controls as Nano, against the `Cosmos3-Super` endpoint.


In [ ]:
v2v_super_payload, v2v_super_output, v2v_super_model = create_payload("v2v_super")


### Run


In [ ]:
check_trtllm_server(v2v_super_model)
run_trtllm_payload(v2v_super_payload, v2v_super_output, model=v2v_super_model)


### View Results


In [ ]:
view_run(v2v_super_output)


# Cosmos3-Edge Examples

Use cases for the compact single-GPU `Cosmos3-Edge` checkpoint. This section is self-contained; you can run it without the Cosmos3-Nano and Cosmos3-Super sections above.

TensorRT-LLM serves Edge for text-to-image, text-to-video, and image-to-video. There is no audio variant (Edge has no audio tower), no action mode (the checkpoint's action weights are not served by this pipeline), and no video-to-video case (validated for Nano and Super only).

Edge runs a different shape from Nano and Super: video requests use 832x480 with 121 frames, 50 UniPC steps and guidance 5.0, and text-to-image uses Edge's native 640x640 with guidance 4.0. These come from `EDGE_VIDEO_SAMPLING` and `EDGE_T2I_SAMPLING` in the helper cell above.

## Edge: Text to Image

Edge text-to-image generation using the same structured JSON prompt as the Nano and Super sections, at Edge's native 640x640.

This case runs **native image mode**: it posts to `/v1/images/generations` with `output_type="image"` in `extra_params` and decodes the base64 PNG from the response. That flag is what selects the image path — without it the server runs video mode, and video mode defaults the negative prompt to Cosmos3's video negative prompt, whose motion and frame-to-frame artifact terms have no meaning for a still. The images API carries no frame or frame-rate fields, so the request sends none.

In [ ]:
t2i_edge_payload, t2i_edge_output, t2i_edge_model = create_payload("t2i_edge")

### Run

In [ ]:
check_trtllm_server(t2i_edge_model)
run_trtllm_payload(t2i_edge_payload, t2i_edge_output, model=t2i_edge_model)

### View Results

In [ ]:
view_run(t2i_edge_output)

## Edge: Text to Video

Edge text-to-video generation using the same structured JSON prompt, at 832x480 with 121 frames. Audio is never enabled for Edge.

In [ ]:
t2v_edge_payload, t2v_edge_output, t2v_edge_model = create_payload("t2v_edge")

### Run

In [ ]:
check_trtllm_server(t2v_edge_model)
run_trtllm_payload(t2v_edge_payload, t2v_edge_output, model=t2v_edge_model)

### View Results

In [ ]:
view_run(t2v_edge_output)

## Edge: Image to Video

Edge image-to-video generation from the coastal-road dashcam frame. This case uses the Edge variant of the prompt asset, whose timeline describes the 5-second, 121-frame Edge clip rather than the longer Nano/Super clip. The reference image is uploaded as multipart `input_reference`.

In [ ]:
i2v_edge_payload, i2v_edge_output, i2v_edge_model = create_payload("i2v_edge")

### Run

In [ ]:
check_trtllm_server(i2v_edge_model)
run_trtllm_payload(i2v_edge_payload, i2v_edge_output, model=i2v_edge_model)

### View Results

In [ ]:
view_run(i2v_edge_output)